# RAG Day 4

## Evaluation!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Keep in mind how you would evaluate RAG for your business</h2>
            <span style="color:#181;">This is such an important part of building an accurate and reliable RAG pipeline. And it's applicable to many aspects of solving business problems with LLMs. People are often focused on RAG architecture and RAG frameworks for their business. But even more important: evaluations!</span>
        </td>
    </tr>
</table>

In [2]:
import zipfile

# Extract evaluation
with zipfile.ZipFile("/content/evaluation.zip", "r") as zip_ref:
    zip_ref.extractall("/")

# Extract implementation
with zipfile.ZipFile("/content/implementation.zip", "r") as zip_ref:
    zip_ref.extractall("/")

# Extract knowledge base
with zipfile.ZipFile("/content/knowledge-base.zip", "r") as zip_ref:
    zip_ref.extractall("/")

print("All three ZIP files extracted successfully!")

All three ZIP files extracted successfully!


In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

/tmp/ipykernel_4261/3216454956.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [5]:
pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.2/122.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1.4.9:
      Successfully uninstalled langchain-core-1.4.9


In [7]:
pip install langchain_chroma


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 737.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [2]:
pip install langchain_huggingface

In [4]:
pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [2]:
# We are using a local, low-cost model through Ollama

MODEL = "llama3.2"
db_name = "vector_db"

from openai import OpenAI

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print(f"Using Ollama model: {MODEL}")

Using Ollama model: llama3.2


In [12]:
# How many characters in all the documents?

knowledge_base_path = "/knowledge-base/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 0 files in the knowledge base
Total characters in knowledge base: 0


In [14]:
# How many tokens in all the documents?

import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode(entire_knowledge_base)

token_count = len(tokens)

print(f"Total tokens: {token_count:,}")

Total tokens: 0


In [15]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("/knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [16]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# Careers at Insurellm

## Why Join Insurellm?

At Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.

After reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.

### Our Culture' metadata={'source': '/knowledge-base/company/careers.md', 'doc_type': 'company'}


In [17]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vectorstore created with 413 documents


In [18]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


In [21]:
!find / -maxdepth 3 -type d -name "evaluation" 2>/dev/null


/evaluation


In [22]:
!find / -maxdepth 3 -type f -name "test.py" 2>/dev/null

/evaluation/test.py


In [23]:
import sys
import os

# Tell Python that / is where our packages are located
if "/" not in sys.path:
    sys.path.insert(0, "/")

# Make evaluation a Python package
open("/evaluation/__init__.py", "a").close()

# Make implementation a Python package too
if os.path.isdir("/implementation"):
    open("/implementation/__init__.py", "a").close()

print("Evaluation path:", "/evaluation")
print("Test file exists:", os.path.exists("/evaluation/test.py"))

Evaluation path: /evaluation
Test file exists: True


In [24]:
from evaluation import test

In [25]:
tests = test.load_tests()

In [26]:
len(tests)

150

In [27]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)


Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [28]:
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'spanning': 20,
         'holistic': 10})

In [135]:
!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"llama3.2:latest","model":"llama3.2:latest","modified_at":"2026-08-08T14:14:59.07860826Z","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"3.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3072},"capabilities":["completion","tools"]}]}

In [136]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [137]:
import time
time.sleep(5)

!ollama list

NAME               ID              SIZE      MODIFIED          
llama3.2:latest    a80c4f17acd5    2.0 GB    About an hour ago    


In [41]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 137 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (410 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [122]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [126]:
!ollama --version

ollama version is 0.32.6


In [111]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [103]:
import time
time.sleep(5)

In [104]:
!curl http://127.0.0.1:11434/api/tags

{"models":[{"name":"llama3.2:latest","model":"llama3.2:latest","modified_at":"2026-08-08T14:06:08.928382323Z","size":2019393189,"digest":"a80c4f17acd55265feec403c7aef86be0c25983ab279d83f3bcd3abbcb5b8b72","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"3.2B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":3072},"capabilities":["completion","tools"]}]}

In [105]:
!ollama pull llama3.2

In [112]:
!ollama list

NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    3 minutes ago    


In [55]:
!ollama run llama3.2 "Who are you?"

I'm an artificial intelligence model known as Llama. Llama stands for "Larg
"Large Language Model Meta AI."



In [30]:
!pip install -q litellm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 665.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.0 MB/s eta 0:00:00


In [57]:
!grep -nE "OpenAI|ChatOpenAI|OpenAIEmbeddings|OPENAI_API_KEY" /implementation/answer.py

2:from langchain_openai import ChatOpenAI, OpenAIEmbeddings
17:embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
31:llm = ChatOpenAI(temperature=0, model_name=MODEL)


In [58]:
!cat /implementation/answer.py

from pathlib import Path
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document

from dotenv import load_dotenv


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"
DB_NAME = str(Path(__file__).parent.parent / "vector_db")

# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
RETRIEVAL_K = 10

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(

In [59]:
!pip install -q langchain-ollama

In [60]:
!ls -lh /vector_db

ls: cannot access '/vector_db': No such file or directory


In [62]:
import sys

for module in list(sys.modules):
    if module.startswith("implementation"):
        del sys.modules[module]

In [63]:
from implementation.answer import answer_question, fetch_context

print("✅ answer.py loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ answer.py loaded


In [64]:
docs = fetch_context("Who is Avery Lancaster?")

print("Number of documents:", len(docs))

Number of documents: 10


In [70]:
for i, doc in enumerate(docs):
    print(f"\n--- DOCUMENT {i+1} ---")
    print(doc.page_content[:500])


--- DOCUMENT 1 ---
## Other HR Notes
- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  
- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  
- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible 

--- DOCUMENT 2 ---
# Avery Lancaster

## Summary
- **Date of Birth**: March 15, 1985
- **Job Title**: Co-Founder & Chief Executive Officer (CEO)
- **Location**: San Francisco, California
- **Current Salary**: $225,000  

## Insurellm Career Progression
- **2015 - Present**: Co-Founder & CEO  
  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadersh

In [61]:
!ls -lh /content/vector_db

total 4.3M
drwxr-xr-x 2 root root 4.0K Aug  8 13:31 bc09d953-45ed-4909-a4aa-bb4a3d448f43
-rw-r--r-- 1 root root 4.3M Aug  8 13:31 chroma.sqlite3


In [117]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [118]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.16666666666666666, ndcg=0.29420493957109245, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [ ]:
eval, answer, chunks = evaluate_answer(example)

In [99]:
eval

<function eval(source, globals=None, locals=None, /)>

In [139]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

AttributeError: 'builtin_function_or_method' object has no attribute 'feedback'

In [140]:
answer_eval, answer, chunks = evaluate_answer(example)


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



InternalServerError: litellm.InternalServerError: InternalServerError: OpenAIException - Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [141]:
import evaluation.eval

print("MODEL =", evaluation.eval.MODEL)
print("OLLAMA =", evaluation.eval.OLLAMA_API_BASE)

MODEL = gpt-4.1-nano


AttributeError: module 'evaluation.eval' has no attribute 'OLLAMA_API_BASE'

In [142]:
import evaluation.eval

print(evaluation.eval.__file__)

/evaluation/eval.py


In [143]:
!grep -n "MODEL\|OLLAMA_API_BASE" /evaluation/eval.py

20:MODEL = "ollama/llama3.2"
21:OLLAMA_API_BASE = "http://127.0.0.1:11434"
27:# RETRIEVAL EVALUATION MODEL
55:# ANSWER EVALUATION MODEL
340:        model=MODEL,
342:        api_base=OLLAMA_API_BASE,


In [144]:
import sys

for module in list(sys.modules):
    if module.startswith("evaluation"):
        del sys.modules[module]

from evaluation.eval import evaluate_answer, evaluate_retrieval

import evaluation.eval

print("MODEL =", evaluation.eval.MODEL)
print("OLLAMA =", evaluation.eval.OLLAMA_API_BASE)

MODEL = ollama/llama3.2
OLLAMA = http://127.0.0.1:11434


In [145]:
from litellm import completion

response = completion(
    model="ollama/llama3.2",
    messages=[
        {
            "role": "user",
            "content": "Reply with only the word OK"
        }
    ],
    api_base="http://127.0.0.1:11434",
)

print(response.choices[0].message.content)

OK


In [146]:
answer_eval, answer, chunks = evaluate_answer(example)

In [147]:
print("ANSWER:")
print(answer)

print("\nDOCUMENTS:", len(chunks))

print("\nFEEDBACK:")
print(answer_eval.feedback)

print("\nACCURACY:", answer_eval.accuracy)
print("COMPLETENESS:", answer_eval.completeness)
print("RELEVANCE:", answer_eval.relevance)

ANSWER:
According to the context, Maxine was recognized as Insurellm Innovator of the Year (IIOTY) in 2023.

DOCUMENTS: 10

FEEDBACK:
Minor correction needed

ACCURACY: 4.0
COMPLETENESS: 5.0
RELEVANCE: 5.0
